# 00 — Limpieza y muestreo

**Rol 4 — Diego Alejandro Sandoval (339271)**

Preparación de datos común a los tres bloques técnicos. Este notebook:
1. Carga y verifica el dataset (`train.csv`, NYC Taxi Trip Duration — Kaggle).
2. Aplica 5 criterios de limpieza (solo errores de medición físicamente imposibles).
3. Verifica que la limpieza no introduce sesgo relevante.
4. Extrae una muestra estratificada proporcional (día×hora) de 50,004 viajes.
5. Verifica la representatividad de la muestra con divergencia KL contra la población.

Genera `data/train_limpio.parquet` (población limpia, 1,438,943 viajes) y
`data/muestra_50k.parquet` (muestra, usada por los Roles 2 y 3).

## 1. Carga y verificación estructural

In [1]:
import pandas as pd, numpy as np

df = pd.read_csv('train.csv', parse_dates=['pickup_datetime', 'dropoff_datetime'])
print(f"Filas: {len(df):,} | Columnas: {list(df.columns)}")
print(f"Nulos totales: {df.isnull().sum().sum()}")
print(f"IDs duplicados: {df['id'].duplicated().sum()}")
print(f"Rango de fechas: {df.pickup_datetime.min()} a {df.pickup_datetime.max()}")

# Consistencia: trip_duration debe coincidir con dropoff - pickup
diff = (df.dropoff_datetime - df.pickup_datetime).dt.total_seconds()
print(f"Filas donde trip_duration no coincide con dropoff-pickup: {(diff.round() != df.trip_duration).sum()}")


Filas: 1,458,644 | Columnas: ['id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag', 'trip_duration']
Nulos totales: 0


IDs duplicados: 0
Rango de fechas: 2016-01-01 00:00:17 a 2016-06-30 23:59:39
Filas donde trip_duration no coincide con dropoff-pickup: 0


## 2. Limpieza — 5 criterios de plausibilidad física

**Principio rector: se eliminan errores de medición, no eventos reales.** Los días atípicos
(ventisca de enero, Memorial Day) se conservan y se marcan, no se eliminan.

In [2]:
def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

df['dist_km'] = haversine_km(df.pickup_longitude, df.pickup_latitude, df.dropoff_longitude, df.dropoff_latitude)
df['speed_kmh'] = df.dist_km / (df.trip_duration/3600)

LON, LAT = (-74.3, -73.7), (40.5, 40.9)
fallas = pd.DataFrame({
    'C1_fuera_NYC':   ~(df.pickup_longitude.between(*LON) & df.pickup_latitude.between(*LAT) &
                        df.dropoff_longitude.between(*LON) & df.dropoff_latitude.between(*LAT)),
    'C2_pasajeros':   ~df.passenger_count.between(1, 6),
    'C3_duracion':    ~df.trip_duration.between(60, 3*3600),
    'C4_dist_min':    df.dist_km < 0.1,
    'C5_velocidad':   df.speed_kmh > 100,
})
for c in fallas:
    print(f"  {c:16s} {fallas[c].sum():7,d}  ({fallas[c].mean()*100:.3f}%)")

mask = ~fallas.any(axis=1)
print(f"\nN inicial: {len(df):,} | eliminados: {(~mask).sum():,} ({(1-mask.mean())*100:.3f}%) | N final: {mask.sum():,}")


  C1_fuera_NYC       1,698  (0.116%)
  C2_pasajeros          65  (0.004%)
  C3_duracion       10,707  (0.734%)
  C4_dist_min       13,318  (0.913%)
  C5_velocidad         175  (0.012%)

N inicial: 1,458,644 | eliminados: 19,701 (1.351%) | N final: 1,438,943


In [3]:
# Verificación: la limpieza no sesga relevante la curva horaria de demanda
df['hour'] = df.pickup_datetime.dt.hour
p0 = df.hour.value_counts(normalize=True).sort_index()
p1 = df[mask].hour.value_counts(normalize=True).sort_index()
print(f"Cambio máximo en la distribución horaria tras limpiar: {(p1-p0).abs().max()*100:.3f} puntos porcentuales")

tasa_h = (~mask).groupby(df.hour).mean()*100
print(f"Tasa de eliminación por hora: mínimo {tasa_h.min():.2f}% (h{tasa_h.idxmin()}) | máximo {tasa_h.max():.2f}% (h{tasa_h.idxmax()})")


Cambio máximo en la distribución horaria tras limpiar: 0.014 puntos porcentuales
Tasa de eliminación por hora: mínimo 1.15% (h9) | máximo 2.63% (h4)


In [4]:
# Días atípicos: se marcan, no se eliminan
eventos = pd.to_datetime(['2016-01-23', '2016-01-24', '2016-05-30']).date
df['dia_atipico'] = df.pickup_datetime.dt.date.isin(eventos)

limpio = df[mask].copy()
print(f"Viajes en días atípicos conservados: {limpio.dia_atipico.sum():,} ({limpio.dia_atipico.mean()*100:.3f}% del dataset limpio)")

cols = ['id','vendor_id','pickup_datetime','passenger_count','pickup_longitude','pickup_latitude',
        'dropoff_longitude','dropoff_latitude','store_and_fwd_flag','trip_duration','dia_atipico']
limpio[cols].to_parquet('train_limpio.parquet', index=False)
print("Guardado: train_limpio.parquet")


Viajes en días atípicos conservados: 10,436 (0.725% del dataset limpio)


Guardado: train_limpio.parquet


## 3. Muestreo — estratificado proporcional por día×hora

Justificación completa (comparación de 3 estrategias × 5 tamaños × 5 semillas) en la bitácora
metodológica. Resumen: la estratificada por día×hora hace exacta la distribución temporal (eje
del Rol 1) sin costo, y 50,000 es el punto de rendimientos decrecientes en el error espacial
(la dimensión más difícil de representar: 620 zonas vs. 24 horas).

In [5]:
SEMILLA = 42
poblacion = pd.read_parquet('train_limpio.parquet')
N = len(poblacion)
poblacion['hour'] = poblacion.pickup_datetime.dt.hour
poblacion['dow']  = poblacion.pickup_datetime.dt.dayofweek

muestra = poblacion.groupby(['dow', 'hour'], group_keys=False).sample(frac=50_000/N, random_state=SEMILLA)
print(f"Tamaño de la muestra: {len(muestra):,}")


Tamaño de la muestra: 50,004


In [6]:
# Verificación de representatividad: KL(muestra || población) en bits
def kl_bits(sample_col, pop_col):
    p = sample_col.value_counts(normalize=True)
    q = pop_col.value_counts(normalize=True).reindex(p.index)
    return float((p * np.log2(p/q)).sum())

for col in ['hour', 'dow']:
    print(f"  KL {col:6s}: {kl_bits(muestra[col], poblacion[col]):.5f} bits")

muestra.to_parquet('muestra_50k.parquet', index=False)
print("\nGuardado: muestra_50k.parquet")


  KL hour  : 0.00000 bits
  KL dow   : 0.00000 bits

Guardado: muestra_50k.parquet


## 4. Siguientes pasos

Los notebooks `01`, `02` y `03` parten de `muestra_50k.parquet` (y `01` también valida contra
`train_limpio.parquet`) para calcular las variables derivadas y los resultados de cada bloque
técnico. `04` integra los tres.